In [1]:
# Imports
import struct
import traceback
import math
import os
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import open3d as o3d
from scipy.spatial import Delaunay
from functools import reduce
import copy

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [69]:
# Constants
# Ethernet ----------------------------------------------------------------
SENSOR_FRONT_IP = "192.168.1.10"
SENSOR_RIGHT_IP = "192.168.1.11"
SENSOR_LEFT_IP = "192.168.1.12"
SENSOR_TOP_IP = "192.168.1.13"

# Scans -------------------------------------------------------------------
SCANS_DIRECTORY = r'C:\Users\seu_usuario\seu_path\DAS5104_Projeto_LiDAR\pointcloud'
SCAN_DIRECTION = "ccw"

# Sensors -----------------------------------------------------------------
SENSOR_TOP_HEIGHT = 2400

# [sensor_x_offset, sensor_y_offset, sensor_z_offset]
SENSOR_RIGHT_TRANSLATION = (1070, 1130, 0)
SENSOR_LEFT_TRANSLATION = (1105, -1130, 0)

# Euler angles [x, y, z]
SENSOR_RIGHT_ROTATION = (0, 0, 0)
SENSOR_LEFT_ROTATION = (0, 0, 0)

BOUNDARIES_PROFILE_X_MIN = 100
BOUNDARIES_PROFILE_X_MAX = 2385
BOUNDARIES_PROFILE_Y_MIN = -1000
BOUNDARIES_PROFILE_Y_MAX = 1000
BOUNDARIES_PROFILE_Z_MIN = -2750
BOUNDARIES_PROFILE_Z_MAX = -500

BOUNDARIES_ZAXIS_X_MIN = -2300
BOUNDARIES_ZAXIS_X_MAX = -400
BOUNDARIES_ZAXIS_Y_MIN = -7000
BOUNDARIES_ZAXIS_Y_MAX = -100

# Output -----------------------------------------------------------------
OUTPUT_DIRECTORY = r'outputs'

TRUCK_VELOCITY = 10

In [70]:
def ntp64_to_seconds(integer):
    # Upper 32 bits for seconds
    seconds = integer >> 32

    # Lower 32 bits for fractional seconds
    fractional_seconds = integer & 0xFFFFFFFF
    fractional_seconds = fractional_seconds / 0x100000000

    return round(seconds + fractional_seconds, 3)

In [71]:
def polar_to_xy(distances: list, first_angle: int, angular_increment: int):
    first_angle /= 10000
    angular_increment /= 10000

    xy = list()

    for i, distance in enumerate(distances):
        # Invalid measurements return 0xFFFFFFFF
        if distance == 4_294_967_295:
            continue

        angle = (first_angle + i * angular_increment) * math.pi / 180.0

        x = round(distance * math.cos(angle))
        y = round(distance * math.sin(angle))

        xy.append((x, y))

    return xy

In [72]:
def process_binary_file(file_path: str):
    file = open(file_path, "rb")
    data = file.read()
    file.close()

    scans = dict()
    magic_byte = struct.pack("H", 0xa25c)

    for packet in data.split(magic_byte):

        if len(packet) <= 10:
            continue

        try:
            packet_size = struct.unpack("I", packet[2:6])[0] - len(magic_byte)
            header_size = struct.unpack("H", packet[6:8])[0] - len(magic_byte)
            scan_number = struct.unpack("H", packet[8:10])[0]
            timestamp_raw = struct.unpack("Q", packet[12:20])[0]
            first_angle = struct.unpack("i", packet[42:46])[0]
            angular_increment = struct.unpack("i", packet[46:50])[0]
        except Exception as e:
            print(f"Package from IP {os.path.basename(file_path).split('.bin')[0]} corrupted...")
            print(e)
            continue

        if len(packet) != packet_size:
            print(f"Package from IP {os.path.basename(file_path).split('.bin')[0]} corrupted")
            print('Expected packet size:', packet_size)
            print('Received packet size:', len(packet))
            continue

        if scan_number not in scans:
            scans[scan_number] = dict()
            scans[scan_number]["xy"] = list()
            scans[scan_number]["timestamp"] = ntp64_to_seconds(timestamp_raw)

        payload = packet[header_size:]  # list[uint32] - 4byte
        distances = struct.unpack(f"{len(payload) // 4}I", payload[:len(payload) // 4 * 4])

        scans[scan_number]["xy"].extend(polar_to_xy(distances, first_angle, angular_increment))

    return scans

In [73]:
def calculate_z_axis(scans_front: dict, x_min: int, x_max: int, y_min: int, y_max: int, velocity: int):
    z_axis = {}
    xyz_front = []

    for i, scan_key in enumerate(sorted(scans_front.keys())):
        z_axis[i] = y_min

        for xy in scans_front[scan_key]["xy"]:
            x = xy[0]
            y = xy[1]
            z = i * velocity

            if x <= x_min or x >= x_max or y <= y_min or y >= y_max:
                continue

            if y > z_axis[i]:
                z_axis[i] = y

            xyz_front.append((x, y, z))

    return z_axis, xyz_front

In [74]:
def reconstruct_z_axis(scans: dict, z_axis: dict) -> list[tuple[int, int, int]]:
    xyz = list()

    for key in zip(sorted(scans.keys()), sorted(z_axis.keys())):
        for xy in scans[key[0]]["xy"]:
            x = xy[0]
            y = xy[1]
            z = z_axis[key[1]]

            xyz.append((x, y, z))

    return xyz

In [75]:
def transform(points, rotation: tuple[int, int, int], translation: tuple[int, int, int]):
    try:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points)
    
        if rotation != (0, 0, 0):
            rotation_matrix = pcd.get_rotation_matrix_from_xyz(rotation)
            pcd.rotate(rotation_matrix, center=(0, 0, 0))

        if translation != (0, 0, 0):
            pcd.translate(translation)
    except Exception as e:
        print(f'Error in point {points}: {traceback.format_exc()}')

    return np.asarray(pcd.points)

In [76]:
def remove_boundaries(points, x_min: int, x_max: int, y_min: int, y_max: int):
  return [p for p in points if not (p[0] <= x_min or p[0] >= x_max or p[1] <= y_min or p[1] >= y_max)]

In [77]:
def filter_point_cloud(points, voxel_size, nb_neighbors, std_ratio, nb_points, radius):
  pcd = o3d.geometry.PointCloud()
  pcd.points = o3d.utility.Vector3dVector(points)

  xyz_1 = pcd.voxel_down_sample(voxel_size)
  # xyz_3 = pcd.uniform_down_sample(every_k_points=2)
  xyz_2, _ = xyz_1.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)
  xyz_3, _ = xyz_2.remove_radius_outlier(nb_points=nb_points, radius=radius)

  return np.asarray(xyz_3.points)

In [78]:
def get_point_cloud(obj, velocity):
  scans_front = process_binary_file(os.path.join(SCANS_DIRECTORY, obj, f'{SENSOR_FRONT_IP}.bin'))
  scans_right = process_binary_file(os.path.join(SCANS_DIRECTORY, obj, f'{SENSOR_RIGHT_IP}.bin'))
  scans_left = process_binary_file(os.path.join(SCANS_DIRECTORY, obj, f'{SENSOR_LEFT_IP}.bin'))
  scans_top = process_binary_file(os.path.join(SCANS_DIRECTORY, obj, f'{SENSOR_TOP_IP}.bin'))
  
  z_axis, _ = calculate_z_axis(
    scans_front,
    BOUNDARIES_ZAXIS_X_MIN,
    BOUNDARIES_ZAXIS_X_MAX,
    BOUNDARIES_ZAXIS_Y_MIN,
    BOUNDARIES_ZAXIS_Y_MAX,
    velocity
  )
  
  xyz_right = reconstruct_z_axis(scans_right, z_axis)
  # plot_point_cloud(xyz_right, 'Scans do sensor right com reconstrução do eixo Z')
  xyz_left = reconstruct_z_axis(scans_left, z_axis)
  xyz_top = reconstruct_z_axis(scans_top, z_axis)
  
  xyz_right = transform(xyz_right, SENSOR_RIGHT_ROTATION, SENSOR_RIGHT_TRANSLATION)
  # plot_point_cloud(xyz_right, 'Scans do sensor right após translação')
  xyz_left = transform(xyz_left, SENSOR_LEFT_ROTATION, SENSOR_LEFT_TRANSLATION)
  
  xyz_right = remove_boundaries(
    xyz_right,
    BOUNDARIES_PROFILE_X_MIN,
    BOUNDARIES_PROFILE_X_MAX,
    BOUNDARIES_PROFILE_Y_MIN,
    BOUNDARIES_PROFILE_Y_MAX,
  )
  
  # plot_point_cloud(xyz_right, 'Scans do sensor right sem dados fora dos limites')

  xyz_left = remove_boundaries(
    xyz_left,
    BOUNDARIES_PROFILE_X_MIN,
    BOUNDARIES_PROFILE_X_MAX,
    BOUNDARIES_PROFILE_Y_MIN,
    BOUNDARIES_PROFILE_Y_MAX,
  )

  xyz_top = remove_boundaries(
    xyz_top,
    BOUNDARIES_PROFILE_X_MIN,
    BOUNDARIES_PROFILE_X_MAX,
    BOUNDARIES_PROFILE_Y_MIN,
    BOUNDARIES_PROFILE_Y_MAX,
  )

  xyz_right = filter_point_cloud(xyz_right, 1, 40, 0.1, 25, 100)
  # plot_point_cloud(xyz_right, 'Scans do sensor right com filtros')
  xyz_left = filter_point_cloud(xyz_left, 1, 40, 0.1, 25, 100)
  xyz_top = filter_point_cloud(xyz_top, 1, 60, 0.06, 25, 120)
  
  xyz = list()
  xyz.extend(xyz_right)
  xyz.extend(xyz_left)
  xyz.extend(xyz_top)

  xyz = transform(xyz, (0, 0, -math.pi/2), (0, SENSOR_TOP_HEIGHT, 0))
  # plot_point_cloud(xyz, 'Point cloud reconstruída')

  return xyz

In [79]:
def get_bucket_bottom_height(point_cloud):
  n_points = {}
  for p in point_cloud:
    height = round(p[1])
    if height in n_points:
      n_points[height] += 1
    else:
      n_points[height] = 1

  return max(n_points, key=n_points.get)

In [80]:
empty_obj = 'carro_vazio_2024-05-06_11h22min41s'
empty_point_cloud = get_point_cloud(empty_obj, 5)
bucket_bottom_y = get_bucket_bottom_height(empty_point_cloud)
empty_point_cloud = [p for p in empty_point_cloud if -2750 < p[2] < -500 and p[1] >= bucket_bottom_y - 10]

full_obj = 'caixa_brita_uniforme_2024-05-06_11h33min44s'
full_point_cloud = get_point_cloud(full_obj, 5)
full_point_cloud = [p for p in full_point_cloud if -2750 < p[2] < -500 and p[1] >= bucket_bottom_y - 10]

Package from IP 192.168.1.12 corrupted...
unpack requires a buffer of 8 bytes
Package from IP 192.168.1.12 corrupted
Expected packet size: -2
Received packet size: 1385
Package from IP 192.168.1.13 corrupted
Expected packet size: 1402
Received packet size: 59
Package from IP 192.168.1.13 corrupted
Expected packet size: 3
Received packet size: 1341


In [81]:
np.random.rand(5, 3)

array([[0.82326163, 0.12607805, 0.02631191],
       [0.05869754, 0.30918517, 0.30097991],
       [0.32897148, 0.1562417 , 0.48300469],
       [0.04907893, 0.13934418, 0.04200057],
       [0.56790633, 0.28133598, 0.13708859]])

In [82]:
import open3d as o3d
import numpy as np

test_point_cloud = full_point_cloud[:]

# Load or create a point cloud
# Here we create a sample point cloud for demonstration purposes
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(test_point_cloud)

# Define the origin point for the rays
# origin = np.array([0, 0, 0])
origin = np.array([11.5, 1000, -1800])

# Get the points from the point cloud
points = np.asarray(pcd.points)

# Visualize the point cloud and the rays
lines = []
for i in range(points.shape[0]):
    lines.append([0, i + 1])

line_set = o3d.geometry.LineSet(
    points=o3d.utility.Vector3dVector(np.vstack((origin, points))),
    lines=o3d.utility.Vector2iVector(lines),
)

# # Add color to lines for better visualization
# colors = [[1, 0, 0] for i in range(len(lines))]
# line_set.colors = o3d.utility.Vector3dVector(colors)

# Visualize
o3d.visualization.draw_geometries([pcd, line_set])

full_point_cloud # empty_point_cloud
test_point_cloud # Parte da full
points # Pontos da point cloud
direction_vectors # Vetores de direcao
unitary_direction_vectors # Vetores de direcao mas unitários
rays_points # Raios formados por [x, y, z, x_vetor_direcao, y_vetor_direcao, z_vetor_direcao]
points_render # Pontos no fim dos vetores unitários (transladando os vetores unitários pra origem com origin + unitary_direction_vectors)
point_cloud_render # PCD do points_render
rays_unit # Raios formados por [x, y, z, x_vetor_direcao, y_vetor_direcao, z_vetor_direcao]

In [83]:
import open3d as o3d
import numpy as np

test_point_cloud = empty_point_cloud[:]
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(test_point_cloud)
points = np.asarray(pcd.points)

origin = np.array([11.5, 1000, -1800])


direction_vectors = points - origin
magnitudes = np.linalg.norm(direction_vectors, axis=1)
unitary_direction_vectors = direction_vectors / magnitudes[:, np.newaxis]
rays_points = np.hstack([[origin]*len(unitary_direction_vectors), unitary_direction_vectors])

print(points[:2])
print(unitary_direction_vectors[:2])
print(rays_points[:2])

[[  394.   771. -1962.]
 [  388.   662. -2340.]]
[[ 0.80639648 -0.48278378 -0.34153263]
 [ 0.50878553 -0.45675832 -0.72973223]]
[[ 1.15000000e+01  1.00000000e+03 -1.80000000e+03  8.06396483e-01
  -4.82783777e-01 -3.41532628e-01]
 [ 1.15000000e+01  1.00000000e+03 -1.80000000e+03  5.08785526e-01
  -4.56758321e-01 -7.29732228e-01]]


In [84]:
origin

array([   11.5,  1000. , -1800. ])

In [85]:
points_render = np.vstack([origin, unitary_direction_vectors])
points_render

array([[ 1.15000000e+01,  1.00000000e+03, -1.80000000e+03],
       [ 8.06396483e-01, -4.82783777e-01, -3.41532628e-01],
       [ 5.08785526e-01, -4.56758321e-01, -7.29732228e-01],
       ...,
       [-2.71571116e-01, -5.54926383e-01,  7.86324257e-01],
       [-2.51045592e-01, -5.65254042e-01,  7.85788762e-01],
       [-1.89601956e-01, -5.52775382e-01,  8.11474261e-01]])

In [86]:
rays = o3d.core.Tensor(rays_points,
                       dtype=o3d.core.Dtype.Float32)

scene = o3d.t.geometry.RaycastingScene()
ans = scene.cast_rays(rays)
points_render = origin + unitary_direction_vectors

lines = []
for i in range(points_render.shape[0]):
    lines.append([0, i + 1])

line_set = o3d.geometry.LineSet(
    points=o3d.utility.Vector3dVector(np.vstack((origin, points_render))),
    lines=o3d.utility.Vector2iVector(lines),
)
point_cloud_render = o3d.geometry.PointCloud()
point_cloud_render.points = o3d.utility.Vector3dVector(points_render)
o3d.visualization.draw_geometries([pcd, point_cloud_render, line_set])

In [87]:
plane = o3d.geometry.TriangleMesh.create_box(width=10, height=0.1, depth=10)
plane.paint_uniform_color([0.7, 0.7, 0.7])  # Gray color
plane.translate([11.5 - 5, 1000 - 1.5, -1800])  # Translate the plane to be under the point cloud

o3d.visualization.draw_geometries([point_cloud_render, plane])

In [88]:
rays_unit = np.hstack([[origin]*len(unitary_direction_vectors), unitary_direction_vectors])
rays_unit

array([[ 1.15000000e+01,  1.00000000e+03, -1.80000000e+03,
         8.06396483e-01, -4.82783777e-01, -3.41532628e-01],
       [ 1.15000000e+01,  1.00000000e+03, -1.80000000e+03,
         5.08785526e-01, -4.56758321e-01, -7.29732228e-01],
       [ 1.15000000e+01,  1.00000000e+03, -1.80000000e+03,
         3.58669621e-01, -3.18396282e-01, -8.77484992e-01],
       ...,
       [ 1.15000000e+01,  1.00000000e+03, -1.80000000e+03,
        -2.71571116e-01, -5.54926383e-01,  7.86324257e-01],
       [ 1.15000000e+01,  1.00000000e+03, -1.80000000e+03,
        -2.51045592e-01, -5.65254042e-01,  7.85788762e-01],
       [ 1.15000000e+01,  1.00000000e+03, -1.80000000e+03,
        -1.89601956e-01, -5.52775382e-01,  8.11474261e-01]])

In [89]:
plane_legacy = o3d.t.geometry.TriangleMesh.from_legacy(plane)

# Create a scene and add the triangle mesh
scene = o3d.t.geometry.RaycastingScene()
plane_id = scene.add_triangles(plane_legacy)

rays = o3d.core.Tensor(rays_unit,
                       dtype=o3d.core.Dtype.Float32)

ans = scene.cast_rays(rays)

In [90]:
rays_unit[:1]

array([[ 1.15000000e+01,  1.00000000e+03, -1.80000000e+03,
         8.06396483e-01, -4.82783777e-01, -3.41532628e-01]])

In [91]:
np.asarray(point_cloud_render.points)[:1]

array([[   12.30639648,   999.51721622, -1800.34153263]])

In [92]:
o3d.visualization.draw_geometries([point_cloud_render, plane])

In [93]:
t_hit = ans['t_hit'].isfinite().numpy()
print(f"Number of True values: {np.count_nonzero(t_hit)}")
print(f"Number of False values: {np.count_nonzero(~t_hit)}")

Number of True values: 5492
Number of False values: 9792


In [94]:
pcd_caixa_brita = o3d.io.read_point_cloud("caixa_brita_uniforme_filtrada.ply")
o3d.visualization.draw_geometries([point_cloud_render, plane, pcd_caixa_brita])

In [95]:
pcd_caixa_brita = o3d.io.read_point_cloud("caixa_brita_pico_filtrada.ply")
o3d.visualization.draw_geometries([point_cloud_render, plane, pcd_caixa_brita])

In [96]:
# reset colors
colors = ['blue']*len(pcd.points)

In [97]:
fig = go.Figure()

pcd_empty = o3d.geometry.PointCloud()
pcd_empty.points = o3d.utility.Vector3dVector(empty_point_cloud)
pcd_empty_points = np.asarray(pcd_empty.points)
x, y, z = np.transpose(pcd_empty_points)
fig.add_trace(go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers',
    marker=dict(
        size=3,
        opacity=0.8,
        color=colors
    ),
    name='Caixa Vazia'
))

x, y, z = np.transpose(pcd_caixa_brita.points)
fig.add_trace(go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers',
    marker=dict(
        size=3,
        opacity=0.8
    ),
    name='Caixa Brita'
))

# Update the layout
fig.update_layout(
     legend=dict(
        x=0.8,  # Adjust x-coordinate
        y=0.8,  # Adjust y-coordinate
        traceorder="normal",
        font=dict(
            family="sans-serif",
            size=12,
            color="black"
        ),
        bgcolor="LightSteelBlue",
        bordercolor="Black",
        borderwidth=2
    ),
    margin=dict(l=0, r=0, b=0, t=0),
    scene=dict(
        xaxis=dict(title='X'),
        yaxis=dict(title='Y'),
        zaxis=dict(title='Z')
    )
)

# Show the figure
fig.show()

# Final

In [196]:
def preprocess_point_cloud(pcd, voxel_size, max_nn_normals, max_nn_fpfh):
    # print(":: Downsample with a voxel size %.3f." % voxel_size)
    pcd_down = pcd.voxel_down_sample(voxel_size)

    radius_normal = voxel_size * 2
    # print(":: Estimate normal with search radius %.3f." % radius_normal)
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=max_nn_normals))

    radius_feature = voxel_size * 5
    # print(":: Compute FPFH feature with search radius %.3f." % radius_feature)
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=max_nn_fpfh))
    return pcd_down, pcd_fpfh

In [197]:
def ransac_registration(source_down, target_down, source_fpfh, target_fpfh, voxel_size, max_iteration, confidence):
  distance_threshold = voxel_size * 1.5
  result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down, target_down, source_fpfh, target_fpfh, True,
    distance_threshold,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    3, [
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(
            0.1),
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(
            distance_threshold)
    ], o3d.pipelines.registration.RANSACConvergenceCriteria(max_iteration, confidence))
  
  return result

In [198]:
def icp_registration(source, target, trans_init, voxel_size, method='point_to_point', epsilon=1e-4, max_iteration=30):
  distance_threshold = voxel_size * 0.4
  target.estimate_normals()

  convergence_criteria = o3d.pipelines.registration.ICPConvergenceCriteria(
    relative_fitness=1e-6,
    relative_rmse=1e-6,
    max_iteration=max_iteration)
  if method == 'generalized':
    print(":: Generalized ICP registration is applied on original point")
    estimation_method = o3d.pipelines.registration.TransformationEstimationForGeneralizedICP(epsilon)
    result = o3d.pipelines.registration.registration_generalized_icp(
      source,
      target,
      distance_threshold,
      trans_init,
      estimation_method,
      convergence_criteria)
    return result
  if method == 'point_to_point':
    print(":: Point-to-point ICP registration is applied on original point")
    estimation_method = o3d.pipelines.registration.TransformationEstimationPointToPoint()
  elif method == 'point_to_plane':
    print(":: Point-to-plane ICP registration is applied on original point")
    estimation_method = o3d.pipelines.registration.TransformationEstimationPointToPlane()
  else:
    raise ValueError("Informed method not allowed. Available methods: 'point_to_point', 'point_to_plane', 'generalized'.")
  
  print("   clouds to refine the alignment. This time we use a strict")
  print("   distance threshold %.3f." % distance_threshold)

  result = o3d.pipelines.registration.registration_icp(
    source, target, distance_threshold, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    convergence_criteria
  )
  
  return result

In [199]:
def align_truck_bucket_and_load(load, bucket, voxel_size, max_iteration_ransac, confidence, max_nn_normals, max_nn_fpfh, epsilon, max_iteration_icp, ransac_loop_size=5):
  result_ransac = None
  print(":: RANSAC registration on downsampled point clouds.")
  print("   Since the downsampling voxel size is %.3f," % voxel_size)
  print("   we use a liberal distance threshold %.3f." % (voxel_size*1.5))

  for _ in range(ransac_loop_size):
    source_down, source_fpfh = preprocess_point_cloud(load, voxel_size, max_nn_normals, max_nn_fpfh)
    target_down, target_fpfh = preprocess_point_cloud(bucket, voxel_size, max_nn_normals, max_nn_fpfh)
      
    result = ransac_registration(source_down, target_down, source_fpfh, target_fpfh, voxel_size, max_iteration_ransac, confidence)
    if not result_ransac or result.fitness > result_ransac.fitness:
      result_ransac = result
    
  print('RANSAC result:', result_ransac)
  print('RANSAC transformation:', result_ransac.transformation)
  result_icp = icp_registration(load, bucket, result_ransac.transformation, voxel_size, 'generalized', epsilon, max_iteration_icp)

  print('ICP result:', result_icp)
  print('ICP transformation:', result_icp.transformation)

  aligned = copy.deepcopy(load)
  aligned.transform(result_icp.transformation)
  
  return result_ransac, result_icp, aligned

In [208]:
bucket_dir = 'caixa_vazia'
bucket_points = get_point_cloud(bucket_dir, 5)
bucket_points = [p for p in bucket_points if -2750 < p[2] < -500]

voxel_size = 30
max_nn_normals = 30
max_nn_fpfh = 100
confidence = 1.0
max_iteration_ransac = 1000000
epsilon = 1e-6
max_iteration_icp = 50
ransac_loop_size = 5

# NEED PROCESSING
dirs = ['caixa_brita_uniforme'] # Process one by one

for load_dir in dirs:
  print(load_dir)
  # PROCESSING
  load_points = get_point_cloud(load_dir, 5)
  load_points = [p for p in load_points if -2750 < p[2] < -500]
  
  load = o3d.geometry.PointCloud()
  load.points = o3d.utility.Vector3dVector(load_points)
  bucket = o3d.geometry.PointCloud()
  bucket.points = o3d.utility.Vector3dVector(bucket_points)
  result_ransac, result_icp, aligned = align_truck_bucket_and_load(load, bucket, voxel_size, max_iteration_ransac, confidence, max_nn_normals, max_nn_fpfh,
                              epsilon, max_iteration_icp, ransac_loop_size)

  updated_transformation = np.array(result_icp.transformation)

  # Cancel x rotation
  updated_transformation[1][0] = 0
  updated_transformation[2][0] = 0
  updated_transformation[2][1] = 0
  updated_transformation[1][2] = 0
  updated_transformation[1][1] = 1
  updated_transformation[2][2] = 1

  # Cancel y and z translations
  updated_transformation[1][3] = 0
  updated_transformation[2][3] = 0

  transformed = copy.deepcopy(load)

  transformed.transform(updated_transformation)
  
  kd_tree = o3d.geometry.KDTreeFlann(bucket)
  inner_load_points = []
  threshold_distance = 30
  nb_neighbors = 40
  std_ratio = 1
  nb_points = 20
  radius = 100

  for point in transformed.points:
      [_, idx, _] = kd_tree.search_knn_vector_3d(point, 1)
      closest_point = bucket.points[idx[0]]
      if np.linalg.norm(np.array(point) - np.array(closest_point)) > threshold_distance:
          inner_load_points.append(point)

  removed_points = o3d.geometry.PointCloud()
  removed_points.points = o3d.utility.Vector3dVector(inner_load_points)

  inner_load, _ = removed_points.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)
  inner_load, _ = inner_load.remove_radius_outlier(nb_points=nb_points, radius=radius)  

  '''
  RAYCASTING SECTION BEGIN
  '''

  bucket_points = np.asarray(bucket.points)
  inner_load_points = np.asarray(inner_load.points)

  # Define origin of the ray casting
  origin = np.array([11.5, 1000, -1800])

  # Define direction vector
  direction_vectors = bucket_points - origin
  magnitudes = np.linalg.norm(direction_vectors, axis=1)
  unitary_direction_vectors = direction_vectors / magnitudes[:, np.newaxis]
  rays_unit = np.hstack([[origin]*len(unitary_direction_vectors), unitary_direction_vectors])

  # Generate simple mash to help finding the points above load surface
  radius = 100
  max_nn = 100
  k = 5

  inner_load.estimate_normals(
      search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=radius, max_nn=max_nn))
  inner_load.orient_normals_consistent_tangent_plane(k)

  mesh_real, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(inner_load, depth=2) 
  mesh_real = mesh_real.filter_smooth_simple(number_of_iterations=1)
  mesh_real.paint_uniform_color([0.7, 0.7, 0.7])

  mesh_real_legacy = o3d.t.geometry.TriangleMesh.from_legacy(mesh_real)

  # Create a scene and add the triangle mesh
  scene = o3d.t.geometry.RaycastingScene()
  plane_id = scene.add_triangles(mesh_real_legacy)

  rays = o3d.core.Tensor(rays_unit,
                        dtype=o3d.core.Dtype.Float32)

  ans = scene.cast_rays(rays)

  t_hit = ans['t_hit'].isfinite().numpy()
  print(f"Number of True values: {np.count_nonzero(t_hit)}")
  print(f"Number of False values: {np.count_nonzero(~t_hit)}")

  direction_magnitudes = [np.sqrt(x**2 + y**2 + z**2) for x, y, z in direction_vectors]
  colors = np.array(['green' if hit_distance < direction_magnitudes[i] else 'blue' for i, hit_distance in enumerate(ans['t_hit'].numpy())])
  print(f"Number of Green values: {np.count_nonzero(colors == 'green')}")
  print(f"Number of Blue values: {np.count_nonzero(colors == 'blue')}")

  points_caixa_brita = inner_load.points
  points_to_select = colors == 'green'
  points_empty_base = np.asarray(bucket.points)
  points_empty_base = list(points_empty_base[points_to_select])
  points_caixa_brita.extend(points_empty_base)
  pcd = o3d.geometry.PointCloud()
  pcd.points = o3d.utility.Vector3dVector(points_caixa_brita)
          
  '''
  RAYCASTING SECTION END
  '''

  nb_neighbors = 10
  std_ratio = 20

  pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)

  x1 = [p[0] for p in pcd.points]
  y1 = [p[1] for p in pcd.points]
  z1 = [p[2] for p in pcd.points]

  for i, alpha in enumerate(range(50, 301, 10)):
    for j, n_iterations in enumerate(range(1, 21)):
      it = j + i*20 + 1
      print(f'{it}/{25*20}')
      mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
      bbox = pcd.get_axis_aligned_bounding_box()
      mesh = mesh.crop(bbox)

      # Refine the mesh
      mesh = mesh.filter_smooth_simple(number_of_iterations=n_iterations)
      mesh.paint_uniform_color([0.7, 0.7, 0.7])
      mesh.compute_triangle_normals()

      # Save and visualize the mesh
      # o3d.visualization.draw_geometries([mesh, pcd], mesh_show_wireframe=True, mesh_show_back_face=True)
      o3d.io.write_triangle_mesh(f"mesh_reconstruction_maurici/{load_dir}/alpha{alpha}_it{n_iterations}.ply", mesh)

Package from IP 192.168.1.12 corrupted...
unpack requires a buffer of 8 bytes
Package from IP 192.168.1.12 corrupted
Expected packet size: -2
Received packet size: 1385
caixa_brita_uniforme
Package from IP 192.168.1.13 corrupted
Expected packet size: 1402
Received packet size: 59
Package from IP 192.168.1.13 corrupted
Expected packet size: 3
Received packet size: 1341
:: RANSAC registration on downsampled point clouds.
   Since the downsampling voxel size is 30.000,
   we use a liberal distance threshold 45.000.
RANSAC result: RegistrationResult with fitness=8.738642e-01, inlier_rmse=2.104392e+01, and correspondence_set size of 5674
Access transformation to get result.
RANSAC transformation: [[ 9.99676964e-01  1.26458322e-02  2.20465610e-02 -1.09036156e+02]
 [-1.20548678e-02  9.99569862e-01 -2.67351991e-02 -1.76134559e+02]
 [-2.23751668e-02  2.64607943e-02  9.99399409e-01  2.66044730e+01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
:: Generalized ICP registratio

# Volume

In [117]:
def _new_volume_under_triangle(triangle):

    p1, p2, p3 = triangle
    x1, z1, y1 = p1
    x2, z2, y2 = p2
    x3, z3, y3 = p3

    return (((-x3*y2*z1) + (x2*y3*z1) + (x3*y1*z2) + (-x1*y3*z2) + (-x2*y1*z3) + (x1*y2*z3))/6)

def _get_triangles_vertices(triangles, vertices):
    triangles_vertices = []

    for triangle in triangles:
        new_triangles_vertices = [vertices[triangle[0]], vertices[triangle[1]], vertices[triangle[2]]]
        triangles_vertices.append(new_triangles_vertices)

    return np.array(triangles_vertices)

In [162]:
def delaunay_mesh(mesh_input):
  volume = reduce(lambda a, b:  a + _new_volume_under_triangle(b),
                  _get_triangles_vertices(mesh_input.triangles, mesh_input.vertices), 0)

  return abs(volume)

In [209]:
dirs = ['caixa_brita_uniforme', 'caixa_brita_pico', 'caixa_brita_vao']
results = []

for load_dir in dirs:
  for i, alpha in enumerate(range(50, 301, 10)):
    for j, n_iterations in enumerate(range(1, 21)):
      mesh_input = o3d.io.read_triangle_mesh(f"mesh_reconstruction_maurici/{load_dir}/alpha{alpha}_it{n_iterations}.ply")
      try:
        volume = delaunay_mesh(mesh_input)
        print(f'{alpha} - {n_iterations}: Triangulação de Delaunay (MESH): {volume} mm³ = {volume*1e-6} L')
      except Exception as e:
        volume = 0
        print(f'{alpha} - {n_iterations}: O volume não pode ser estimado através do método da triangulação de delaunay: {e}')
      results.append([load_dir, alpha, n_iterations, volume])

df = pd.DataFrame(results, columns=['load_dir', 'alpha', 'n_iterations', 'volume (mm^3)'])
df

50 - 1: Triangulação de Delaunay (MESH): 400558587.33788884 mm³ = 400.5585873378888 L
50 - 2: Triangulação de Delaunay (MESH): 392606546.1281272 mm³ = 392.6065461281272 L
50 - 3: Triangulação de Delaunay (MESH): 387021950.08231133 mm³ = 387.0219500823113 L
50 - 4: Triangulação de Delaunay (MESH): 382164625.64763147 mm³ = 382.16462564763145 L
50 - 5: Triangulação de Delaunay (MESH): 377874158.73701257 mm³ = 377.8741587370125 L
50 - 6: Triangulação de Delaunay (MESH): 373919036.1867284 mm³ = 373.9190361867284 L
50 - 7: Triangulação de Delaunay (MESH): 370209244.17217755 mm³ = 370.20924417217753 L
50 - 8: Triangulação de Delaunay (MESH): 366679633.8678541 mm³ = 366.6796338678541 L
50 - 9: Triangulação de Delaunay (MESH): 363291884.96958077 mm³ = 363.29188496958074 L
50 - 10: Triangulação de Delaunay (MESH): 360019182.4520167 mm³ = 360.0191824520167 L
50 - 11: Triangulação de Delaunay (MESH): 356843082.2608223 mm³ = 356.84308226082226 L
50 - 12: Triangulação de Delaunay (MESH): 353750150.3

,load_dir,alpha,n_iterations,volume (mm^3)
0,caixa_brita_uniforme,50,1,4.005586e+08
1,caixa_brita_uniforme,50,2,3.926065e+08
2,caixa_brita_uniforme,50,3,3.870220e+08
3,caixa_brita_uniforme,50,4,3.821646e+08
4,caixa_brita_uniforme,50,5,3.778742e+08
...,...,...,...,...
1555,caixa_brita_vao,300,16,8.505474e+07
1556,caixa_brita_vao,300,17,8.638736e+07
1557,caixa_brita_vao,300,18,8.746183e+07
1558,caixa_brita_vao,300,19,8.830920e+07


In [210]:
df.to_excel('resultados_volume_maurici.xlsx', index=False)

# Resultados

In [3]:
# Se quiser usar os df já salvos
df = pd.read_excel('resultados_volume_maurici.xlsx')
df

,load_dir,alpha,n_iterations,volume (mm^3)
0,caixa_brita_uniforme,50,1,4.005586e+08
1,caixa_brita_uniforme,50,2,3.926065e+08
2,caixa_brita_uniforme,50,3,3.870220e+08
3,caixa_brita_uniforme,50,4,3.821646e+08
4,caixa_brita_uniforme,50,5,3.778742e+08
...,...,...,...,...
1555,caixa_brita_vao,300,16,8.505474e+07
1556,caixa_brita_vao,300,17,8.638736e+07
1557,caixa_brita_vao,300,18,8.746183e+07
1558,caixa_brita_vao,300,19,8.830920e+07


In [4]:
df_maurici = df.copy(deep=True)
# df_maurici.rename(columns={'volume': 'volume (mm^3)'}, inplace=True)
df_maurici['volume (L)'] = df_maurici['volume (mm^3)']*1e-6
df_maurici['erro (L)'] = abs(200 - df_maurici['volume (L)'])
df_maurici['erro (%)'] = 100 * df_maurici['erro (L)']/200
df_maurici.sort_values(by='erro (L)')

,load_dir,alpha,n_iterations,volume (mm^3),volume (L),erro (L),erro (%)
60,caixa_brita_uniforme,80,1,2.000008e+08,200.000813,0.000813,0.000406
249,caixa_brita_uniforme,170,10,2.000020e+08,200.002019,0.002019,0.001009
61,caixa_brita_uniforme,80,2,1.998420e+08,199.842018,0.157982,0.078991
653,caixa_brita_pico,110,14,1.997802e+08,199.780195,0.219805,0.109902
614,caixa_brita_pico,90,15,2.002563e+08,200.256331,0.256331,0.128166
...,...,...,...,...,...,...,...
361,caixa_brita_uniforme,230,2,4.164337e+08,416.433661,216.433661,108.216830
1401,caixa_brita_vao,230,2,4.167834e+08,416.783389,216.783389,108.391694
1400,caixa_brita_vao,230,1,4.185402e+08,418.540234,218.540234,109.270117
360,caixa_brita_uniforme,230,1,4.240986e+08,424.098597,224.098597,112.049299


In [5]:
# Melhor parametro em cada tipo de carga
min_value_rows = df_maurici.loc[df_maurici.groupby('load_dir')['erro (L)'].idxmin()]
min_value_rows

,load_dir,alpha,n_iterations,volume (mm^3),volume (L),erro (L),erro (%)
653,caixa_brita_pico,110,14,1.997802e+08,199.780195,0.219805,0.109902
60,caixa_brita_uniforme,80,1,2.000008e+08,200.000813,0.000813,0.000406
1332,caixa_brita_vao,190,13,2.005270e+08,200.527043,0.527043,0.263522


In [17]:
# Melhor parametro pra diminuir o erro global
grouped_df_maurici = df_maurici.groupby(['alpha', 'n_iterations']).sum()
min_grouped_df_maurici = grouped_df_maurici.loc[grouped_df_maurici.groupby('load_dir')['erro (L)'].idxmin()]
min_grouped_df_maurici

,,load_dir,volume (mm^3),volume (L),erro (L),erro (%)
alpha,n_iterations,,,,,
150,18,caixa_brita_uniformecaixa_brita_picocaixa_brit...,5.995700e+08,599.569967,15.48619,7.743095


In [12]:
df = pd.read_excel('resultados_volume_victor.xlsx')
df

,load_dir,alpha,n_iterations,volume
0,caixa_brita_uniforme,50,1,9.897882e+07
1,caixa_brita_uniforme,50,2,1.015978e+08
2,caixa_brita_uniforme,50,3,1.039448e+08
3,caixa_brita_uniforme,50,4,1.051767e+08
4,caixa_brita_uniforme,50,5,1.061131e+08
...,...,...,...,...
1555,caixa_brita_vao,300,16,2.943506e+08
1556,caixa_brita_vao,300,17,2.883784e+08
1557,caixa_brita_vao,300,18,2.824536e+08
1558,caixa_brita_vao,300,19,2.765895e+08


In [13]:
df_victor = df.copy(deep=True)
df_victor.rename(columns={'volume': 'volume (mm^3)'}, inplace=True)
df_victor['volume (L)'] = df_victor['volume (mm^3)']*1e-6
df_victor['erro (L)'] = abs(200 - df_victor['volume (L)'])
df_victor['erro (%)'] = 100 * df_victor['erro (L)']/200
df_victor.sort_values(by='erro (L)')

,load_dir,alpha,n_iterations,volume (mm^3),volume (L),erro (L),erro (%)
1176,caixa_brita_vao,110,17,1.999560e+08,199.956019,0.043981,0.021991
902,caixa_brita_pico,240,3,2.002825e+08,200.282457,0.282457,0.141228
761,caixa_brita_pico,170,2,1.996963e+08,199.696273,0.303727,0.151864
781,caixa_brita_pico,180,2,2.004526e+08,200.452552,0.452552,0.226276
463,caixa_brita_uniforme,280,4,1.994488e+08,199.448751,0.551249,0.275624
...,...,...,...,...,...,...,...
564,caixa_brita_pico,70,5,4.236914e+08,423.691413,223.691413,111.845707
563,caixa_brita_pico,70,4,4.317937e+08,431.793669,231.793669,115.896835
562,caixa_brita_pico,70,3,4.401848e+08,440.184791,240.184791,120.092396
561,caixa_brita_pico,70,2,4.489206e+08,448.920621,248.920621,124.460311


In [14]:
# Melhor parametro em cada tipo de carga
min_value_rows = df_victor.loc[df_victor.groupby('load_dir')['erro (L)'].idxmin()]
min_value_rows

,load_dir,alpha,n_iterations,volume (mm^3),volume (L),erro (L),erro (%)
902,caixa_brita_pico,240,3,2.002825e+08,200.282457,0.282457,0.141228
463,caixa_brita_uniforme,280,4,1.994488e+08,199.448751,0.551249,0.275624
1176,caixa_brita_vao,110,17,1.999560e+08,199.956019,0.043981,0.021991


In [16]:
# Melhor parametro pra diminuir o erro global
grouped_df_victor = df_victor.groupby(['alpha', 'n_iterations']).sum()
min_grouped_df_victor = grouped_df_victor.loc[grouped_df_victor.groupby('load_dir')['erro (L)'].idxmin()]
min_grouped_df_victor

,,load_dir,volume (mm^3),volume (L),erro (L),erro (%)
alpha,n_iterations,,,,,
120,3,caixa_brita_uniformecaixa_brita_picocaixa_brit...,6.210357e+08,621.035735,42.829996,21.414998
